# Part 6 — 오류 분석과 시각화

**목적** 점수 한 줄이 아니라 **무엇을 어떻게 틀리는가**를 눈으로 보여준다.

**무엇을 보나**
1. 단계별(2천 → 2만 → 28만) 오류 신호가 어떻게 변하는가
2. Stage C 후보의 오류 유형 분포 (고정 100건, Claude 1인 판정)
3. 건별 점수 분포 — 평균 뒤에 숨은 0점 무리
4. 검토 후보 사례 원문 비교
5. 구조(인코더-디코더 vs 디코더 전용)·전이학습 비교

**만드는 법**
```
python scripts/error_report.py --runs <A> <B> <C> --labels "Stage A (2천)" ... \
  --error-csv <100건 판정 csv> --stage-error-csv <단계별 30건 csv> --bar "라벨=run" ...
```
지표 정의는 `src/error_probe.py`, 판정 기준은 `docs/03_evaluation.md` 5장에 있다.

> 2026-09-21 검토: 자동 수치는 검토 신호이며 실제 오류율이 아니다. Claude 판정은 사람 검증을 받지 않았다. 최신 해석·그림은 `../docs/07_full_report.md`와 `../artifacts/report_assets/TEST_ERRORS.md`를 따른다. 이 노트북은 저장된 결과만 읽어 표시한다. 학습·추론은 실행하지 않는다.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import Image, display

EVAL = Path("../artifacts/eval")
REP = sorted(EVAL.glob("*_error_report"))[-1]
print("보고 폴더:", REP.name)
pd.set_option("display.max_colwidth", 80)

## 1. 단계별 오류 신호 — 무엇이 줄고 무엇이 늘었나

In [ ]:
t = pd.read_csv(REP / "error_trend.csv")
display(t)
display(Image(str(REP / "01_error_trend.png")))
display(Image(str(REP / "05_quality_trend.png")))

**읽는 법**

- **줄어든 것**: ROUGE-L 0점 비율 19.0 → 13.0%, 10점 미만 25.9 → 20.1%. 자동 점검 통과율 94.3 → 97.3%
- **늘어난 것**: "대화에 없는 내용어" 비율 14.3 → 16.4%. 그런데 **정답 요약은 29.0%**다.
  정상적인 바꿔 쓰기와 환각 모두에서 증가할 수 있다. **정답 평균에 가까워도 품질을 보장하지 않는다**. 원인은 확인하지 않았다.

## 2. 오류 유형 분포 (고정 100건)

In [ ]:
display(Image(str(REP / "02_error_types.png")))
ea = pd.read_csv(sorted(EVAL.glob("*_dev/error_analysis_100.csv"))[-1], encoding="utf-8-sig")
print(ea["labels"].str.split("|").explode().value_counts().to_string())
print("\n오류 1개 이상:", (ea.labels != "무난").sum(), "/ 100")
ea[ea.labels.str.contains("사실 오류")][["no", "labels", "note"]]

**사실 오류 15건의 원인 분류**

| 원인 | 건수 | 예 |
|---|---:|---|
| 인과·수식 대상 오연결 | 6 | 에어컨 때문에 추운데 "날씨가 추워져서 에어컨이 센 것 같다" |
| 주체 뒤바뀜 | 3 | 본인이 소띠라 누나라 부르라 한 것 → "누나가 소띠라고 불렀다" |
| 추측을 단정 | 2 | "~아냐?"라는 추측을 사실처럼 |
| 상태·순서 오류 | 2 | 운동 중인데 "씻고 누워 운동하려고 한다" |
| 의미 왜곡 | 1 | "곱하기밖에 못 쓴다" → "곱하기 때문에 못 쓴다" |
| 고유명사 왜곡 | 1 | 산북동 → 산남동 |

## 3. 건별 점수 분포와 검토 후보 사례

In [ ]:
display(Image(str(REP / "03_rouge_distribution.png")))
display(Image(str(REP / "04_worst_cases.png")))

**0점이 13%나 되는 이유**: ROUGE-L은 정답과 겹치는 단어가 없으면 0이다.
내용이 맞아도 **정답이 고른 부분과 다른 부분**을 요약하면 0점이 된다 (100건 판정에서 3건 확인).
그래서 0점 = 실패가 아니라, **0점 무리 안에 진짜 실패가 섞여 있다**고 읽어야 한다.

## 4. 단계별로 오류 유형이 어떻게 바뀌나 (같은 30건 재판정)

In [ ]:
display(Image(str(REP / "06_error_types_by_stage.png")))
st = pd.read_csv(REP / "error_by_stage_30.csv", encoding="utf-8-sig")
st[(st.A != st.C)][["no", "A", "B", "C"]].head(15)

**정직한 해석**: 유형별 건수는 단조롭게 줄지 않는다(오류 1개 이상 10 → 8 → 11).
30건은 한 유형당 1~2건 차이로 순서가 바뀌는 크기이고 판정자도 Claude 하나이며 사람 검증은 없다.
고정 30건에서만 반복 오류 1 → 0 → 0건을 관찰했다. test에는 어절 3-gram 반복 신호 198건이 있다.
1,500건에서는 일부 자동 지표 개선을 보고하며 **실제 오류 감소는 확인하지 못했다**.

## 5. 구조·전이학습 비교

In [ ]:
display(Image(str(REP / "07_architecture.png")))
print(json.loads((REP / "bar_items.json").read_text(encoding="utf-8")))

- **전이학습**: 같은 구조·같은 데이터인데 사전학습 가중치가 없으면 22.63 → 5.93. 출력도 `아니야 안 것 한다.` 수준으로 무너진다
- **구조**: 인코더-디코더 22.63 vs 디코더 전용 14.71 (+7.9점). 디코더 전용은 반복 붕괴가 심하고(23.6%),
  반복 금지를 켜면 반복은 0%가 되지만 점수는 더 내려간다(13.18) → 이 반복 억제 설정으로 ROUGE가 개선되지 않았다. 원인 관계는 미확인이다
- 한계: 파라미터(275.6M vs 125.2M)·토크나이저·사전학습이 다르다. T5는 best checkpoint, KoGPT2는 마지막 상태를 썼으며, 기존 KoGPT2 실행에는 설정의 label smoothing이 전달되지 않았다. 반복률도 모델별 토큰 단위다. 구조 자체의 효과를 분리하지 못했고, 수정 코드로 재학습하지 않았다.